# 01 - Data exploration
**Project:** E-commerce Sales Dashboard (Online Retail dataset)

**Goal of this notebook:** load the raw data, look at it, and find what is wrong with it. **We do not change anything here.** The fixes are made in `02_cleaning.ipynb`.

## 1. Load the data
The original file is an Excel file, which is slow to read (1 to 2 minutes). The first time, we read it and save a CSV copy. After that, the notebook reads the fast CSV.

> Change the file names below if yours are different. Paths start with `../` because this notebook is inside the `notebooks/` folder.

In [1]:
import os
import pandas as pd

XLSX_PATH = "../data/rawData.xlsx"   # original Excel file
CSV_PATH = "../data/rawData1.csv"    # fast copy

if os.path.exists(CSV_PATH):
    df = pd.read_csv(CSV_PATH)
else:
    df = pd.read_excel(XLSX_PATH)
    df.to_csv(CSV_PATH, index=False)

print("Loaded from:", CSV_PATH if os.path.exists(CSV_PATH) else XLSX_PATH)

Loaded from: ../data/rawData1.csv


## 2. First look
`head()` shows the first 5 rows. `shape` gives `(rows, columns)`. It has no parentheses because it is a stored value, not an action.

In [2]:
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [3]:
print(df.shape)

(541909, 8)


## 3. Column types and missing values
- `info()` lists each column, its type, and how many values are **not** missing.
- `isna().sum()` counts the missing values per column.

In [ ]:
df.info()

In [ ]:
print(df.isna().sum())

## 4. Basic statistics
`describe()` gives the count, mean, std (spread), min, max, and quartiles of the numeric columns. Look at the **min** row for impossible values. `CustomerID` is a label, so ignore its statistics.

In [ ]:
df.describe()

## 5. Looking for problems
### 5.1 Negative quantities
A negative quantity means items were returned or adjusted.

In [ ]:
negative_qty = df[df["Quantity"] < 0]
print("Rows with negative quantity:", len(negative_qty))

### 5.2 Cancelled invoices
According to the dataset documentation, an invoice number starting with `C` is a cancellation. We **check** this rule on our own data instead of trusting it.

In [ ]:
cancelled = df[df["InvoiceNo"].str.startswith("C")]
print("Cancelled invoices (start with C):", len(cancelled))
print("Of these, with negative quantity:", (cancelled["Quantity"] < 0).sum())
cancelled.head()

### 5.3 Negative quantities that are NOT cancellations
`~` means "not", `&` means "and". Each condition needs its own parentheses.

In [ ]:
other_negative = df[(df["Quantity"] < 0) & (~df["InvoiceNo"].str.startswith("C"))]
print("Negative quantity, not cancelled:", len(other_negative))
other_negative.head(10)

### 5.4 Invoice numbers that contain letters
We count how many invoices start with each letter.

In [ ]:
letters = df[df["InvoiceNo"].str.contains("[A-Za-z]")]
print(letters["InvoiceNo"].str[0].value_counts())

### 5.5 Zero or negative prices

In [ ]:
bad_price = df[df["UnitPrice"] <= 0]
print("Rows with price <= 0:", len(bad_price))
bad_price.head()

### 5.6 Duplicate rows
`duplicated()` marks a row True if an identical row appeared before it.

In [ ]:
print("Duplicate rows:", df.duplicated().sum())

### 5.7 Date range
`InvoiceDate` is stored as text. We convert a **copy** to real dates just to look at the range.

In [ ]:
dates = pd.to_datetime(df["InvoiceDate"])
print("First date:", dates.min())
print("Last date: ", dates.max())

### 5.8 Countries

In [ ]:
print(df["Country"].value_counts())

### 5.9 Product codes that are not products
Real products have a stock code that starts with 5 digits (for example `85123A`). Codes like `POST` (postage) or `DOT` (fees) are not products.

In [ ]:
non_products = df[~df["StockCode"].str.match(r"^\d{5}")]
print(non_products["StockCode"].value_counts().head(15))

## 6. Summary of the findings
Fill in the numbers from your own outputs.

| Problem | Rows | Decision (applied in `02_cleaning.ipynb`) |
|---|---|---|
| Rows in the raw data | 541,909 | - |
| Missing `Description` | 1,454 | Removed by the price filter, no separate step |
| Missing `CustomerID` | 135,080 (about 25%) | **Keep** for products, months, and countries. Exclude only for the customer analysis |
| Cancelled invoices (start with `C`) | 9,288 | Remove |
| Negative quantity, not cancelled | 1,336 | Remove |
| Price <= 0 | ... | Remove |
| Duplicate rows | ... | Remove |
| Non-product codes (`POST`, `DOT`...) | ... | Keep in the data, exclude only from the product ranking |
| `InvoiceDate` stored as text | all rows | Convert to a real date |
| Date range | ... to ... | December 2011 is incomplete (data stops on 9 Dec) |

**Next step:** `02_cleaning.ipynb`.